# Reproducibility package paths and historical outputs

Original source: `/data/hulei/HongtaoYu_Lab/Fetal_brain/Revision/GRN/bin100/robustness/04_GRN_rep04.ipynb`.
Original file SHA256: `225bc212a4edcfb6457dc7421001e0d01babe4ec7fda58b9c15da34f09a2f65e`.

**This notebook was not re-executed or retrained for this update. All execution counts and cell outputs below are preserved from the original notebook as historical records. The updated paths have not been runtime-validated.**

Run from the `reproducibility` package root, where `base = Path.cwd()` and `config.json` provides the runtime inputs. By default, the notebook reads the original external input matrix. New training results, temporary files, downsampled matrices, and QC files are written outside the package under `../reproducibility_work`. Original inputs, prior results, and package data are not written back.

Inference parameters, random seeds, constant-gene handling, and AUCell settings are unchanged. The two ranking databases use their exact configured file paths.


# T577 bin100 counts50: rep04
Downsampling seed `20261019`. SpaGRN’s native stepwise workflow is spatial co-expression → modules → cisTarget → AUCell.
The full-data Moran statistic, 30 neighbors, intersection of four FDR criteria, and reference databases are retained. Zero-variance genes are excluded during training; saved results retain all bins and gene columns.
[Official tutorial](https://spagrn.readthedocs.io/en/latest/content/01_Basic_Usage.html)。All visualizations are in the root-level [`T577_SpaGRN_visualization.ipynb`](../../T577_SpaGRN_visualization.ipynb).

In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import scanpy as sc
from spagrn.regulatory_network import InferNetwork

base = Path.cwd()
config = json.loads((base / "config.json").read_text())["training"]
run = config["runs"][3]
out = base / run["output_dir"]
out.mkdir(parents=True, exist_ok=True)
adata = sc.read_h5ad(base / run["input"])
counts = adata.X.astype("float64").tocsc()
variable = np.asarray(counts.power(2).mean(axis=0)).ravel() > np.asarray(counts.mean(axis=0)).ravel() ** 2
analysis = adata[:, variable].copy()
analysis.layers["raw_counts"] = counts[:, variable]
adata.var["spagrn_variable_for_training"] = variable
pd.DataFrame([dict(run=run["run"], seed=run["seed"], bins=adata.n_obs,
                   genes=adata.n_vars, training_genes=analysis.n_vars)])

/data/lvyongji/envs/conda_envs/spagrn/lib/python3.8/site-packages/geopandas/_compat.py:124: UserWarning: The Shapely GEOS version (3.11.4-CAPI-1.17.4) is incompatible with the GEOS version PyGEOS was compiled with (3.10.4-CAPI-1.16.2). Conversions between both will be slow.
  warnings.warn(


,run,seed,bins,genes,training_genes
0,rep04,20261019,117867,15541,15537


In [2]:
reference = config["reference"]
grn = InferNetwork(analysis, project_name=f"T577_SpaGRN_{run['run']}")
grn.tmp_dir = str(out / "tmp_files")
Path(grn.tmp_dir).mkdir(exist_ok=True)
tfs = grn.load_tfs(str(base / reference["tf_list_file"]))
dbs = [database for ranking_file in reference["ranking_files"]
       for database in grn.load_database(ranking_file)]

In [3]:
adj = grn.spg(
    analysis, tf_list=tfs, layer_key="raw_counts", latent_obsm_key="spatial",
    umi_counts_obs_key="total_counts", model="bernoulli", n_neighbors=30,
    weighted_graph=False, jobs=16, local=False, combine=False,
    methods=["FDR_C", "FDR_I", "FDR_G", "FDR"], operation="intersection",
    mode="moran", torch_device="cuda:0", save_tmp=True, cache=False,
    fn=str(out / f"T577_SpaGRN_{run['run']}.adjacencies.csv"),
)

[spg] Entering spg method
[spg] No cached file found, computing new correlations
[spg] No gene_list provided, computing from hotspot
[spg] Creating KNN graph...


[spg] KNN graph created in 21.78 seconds
[spg] Computing autocorrelations...


  0%|          | 0/15537 [00:00<?, ?it/s]

  0%|          | 1/15537 [00:00<4:11:10,  1.03it/s]

  0%|          | 8/15537 [00:01<26:29,  9.77it/s]  

  0%|          | 15/15537 [00:01<13:41, 18.89it/s]

... (2,226 outputs left out)

[spg] Computing global bivariate Moran...


run_in_parallel_m: Total time taken: 8219.1306 seconds


[spg] Local correlations computed in 2.29 hours (17.17 minutes)
[spg] Processing local_correlations...
[spg] Saving local_correlations to file...


[spg] Exiting spg method


In [4]:
matrix = adata.to_df()
modules = grn.get_modules(adj, matrix, rho_mask_dropouts=False, save_tmp=True, cache=False)
regulons = grn.prune_modules(
    modules, dbs, reference["motif_annotation_file"], num_workers=8,
    rank_threshold=1500, auc_threshold=0.05, nes_threshold=3.0,
    motif_similarity_fdr=0.05, save_tmp=True, cache=False, fn=str(base / run["motifs"]),
)

[get_modules] Entering get_modules method
[get_modules] Computing modules from adjacencies...



2026-10-27 00:14:29,001 - pyscenic.utils - INFO - Calculating Pearson correlations.



2026-10-27 00:14:29,427 - pyscenic.utils - WARNING - Note on correlation calculation: the default behaviour for calculating the correlations has changed after pySCENIC verion 0.9.16. Previously, the default was to calculate the correlation between a TF and target gene using only cells with non-zero expression values (mask_dropouts=True). The current default is now to use all cells to match the behavior of the R verision of SCENIC. The original settings can be retained by setting 'rho_mask_dropouts=True' in the modules_from_adjacencies function, or '--mask_dropouts' from the CLI.
	Dropout masking is currently set to [False].



2026-10-27 00:15:41,145 - pyscenic.utils - INFO - Creating modules.


[get_modules] Created 2065 modules in 1.43 minutes (85.94 seconds)
[get_modules] Saving modules to file...


[get_modules] Exiting get_modules method


... (1,981 outputs left out)

[#####################################   ] | 92% Completed | 205.21 s

[#####################################   ] | 92% Completed | 205.31 s

[########################################] | 100% Completed | 205.41 s

Motif enrichment computed in 3.46 minutes (207.54 seconds)
Create regulons from a dataframe of enriched features.
Additional columns saved: []


In [5]:
auc = grn.cal_auc(matrix, regulons, auc_threshold=0.05, num_workers=1,
                 seed=20261014, normalize=False, cache=False, save_tmp=False)
adata.obsm["auc_mtx"] = auc.reindex(adata.obs_names)
adata.uns["adj"] = adj
adata.uns["regulon_dict"] = grn.regulon_dict
adata.uns["method"] = "SpaGRN"
adata.uns["spagrn_reference_species"] = reference["species"]
adata.uns["spagrn_parameters"] = dict(mode="moran", n_neighbors=30,
    methods=["FDR_C", "FDR_I", "FDR_G", "FDR"], operation="intersection",
    rank_threshold=1500, prune_auc_threshold=0.05, nes_threshold=3.0, auc_threshold=0.05)
adata.write_h5ad(base / run["result"], compression="gzip")
adata.obsm["auc_mtx"].to_csv(out / f"T577_SpaGRN_{run['run']}.aucell_all_bins.csv.gz")
pd.DataFrame([dict(run=run["run"], bins=adata.n_obs, genes=adata.n_vars,
    regulons=auc.shape[1], adjacency_rows=len(adj), total_counts=int(adata.obs["total_counts"].sum()))])

  0%|          | 0/218 [00:00<?, ?it/s]

  0%|          | 1/218 [00:03<13:22,  3.70s/it]

  1%|          | 2/218 [00:05<08:16,  2.30s/it]

  1%|▏         | 3/218 [00:05<04:52,  1.36s/it]

  2%|▏         | 4/218 [00:05<03:27,  1.03it/s]

  2%|▏         | 5/218 [00:06<02:41,  1.32it/s]

... (212 outputs left out)

100%|██████████| 218/218 [02:20<00:00,  1.68it/s]

100%|██████████| 218/218 [02:20<00:00,  1.55it/s]

AUCell calculation completed in 3.27 minutes (196.39 seconds)


,run,bins,genes,regulons,adjacency_rows,total_counts
0,rep04,117867,15541,218,4439000,135429514
